# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
from dotenv import load_dotenv
import os
import time

# API key disimpan di file .env (root project), lihat .env.example
load_dotenv('../.env')
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

EMBEDDING_MODEL = os.environ.get('EMBEDDING_MODEL', 'gemini-embedding-001')
LLM_MODEL = os.environ.get('LLM_MODEL', 'gemini-flash-lite-latest')

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


In [2]:
# Cek struktur data
print('Kolom:', list(df.columns))
print('Jumlah kategori:', df['category'].nunique())
print('Missing value:', df.isna().sum().sum())
print()
print(df['category'].value_counts())

Kolom: ['faq_id', 'category', 'question', 'answer']
Jumlah kategori: 8
Missing value: 0

category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5
Name: count, dtype: int64


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

Kategori juga ikut digabungkan supaya topik FAQ lebih jelas.

In [3]:
df['content'] = 'Kategori: ' + df['category'] + '\nQ: ' + df['question'] + '\nA: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Kategori: Pemesanan
Q: Bagaimana cara melakukan pemesanan di platform ini?
A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

Catatan: model `text-embedding-004` sudah tidak tersedia, jadi diganti dengan `gemini-embedding-001`. Semua dokumen dikirim sekaligus dalam satu request (batch) supaya tidak terkena rate limit.

In [4]:
def get_embedding(text, task_type='RETRIEVAL_QUERY'):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

# Embedding semua FAQ sekaligus (batch)
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=documents,
    config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT')
)
embeddings = [e.values for e in result.embeddings]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [5]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)
print('Jumlah entri FAQ:', len(df))
print('Sesuai?', index.ntotal == len(df))

# Simpan index ke file
os.makedirs('../vector_store', exist_ok=True)
faiss.write_index(index, '../vector_store/faq_index.faiss')
print('Index disimpan ke vector_store/faq_index.faiss')

Jumlah vektor tersimpan di index: 59
Jumlah entri FAQ: 59
Sesuai? True
Index disimpan ke vector_store/faq_index.faiss


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

Distance yang dipakai adalah L2, jadi **semakin kecil distance, semakin mirip** query dengan FAQ.

In [6]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'faq_id': df.iloc[idx]['faq_id'],
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': round(float(distances[0][rank]), 4)
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
pd.DataFrame(test_results)[['rank', 'faq_id', 'category', 'question', 'distance']]

,rank,faq_id,category,question,distance
0,1,12,Pengiriman,Bagaimana jika paket saya belum sampai melebih...,0.4028
1,2,14,Pengiriman,Kenapa status pengiriman saya tidak berubah se...,0.4353
2,3,52,Pengiriman,Kenapa nomor resi saya belum bisa dilacak di w...,0.4690


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [7]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"[FAQ #{r['faq_id']}]\nPertanyaan: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''PERSONA:
Kamu adalah asisten customer service e-commerce yang ramah dan membantu. Gunakan Bahasa Indonesia yang sopan dan mudah dipahami.

TASK:
Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ pada bagian CONTEXT. Jangan menggunakan pengetahuan di luar FAQ dan jangan mengarang informasi.
Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan pengguna untuk menghubungi customer service.

CONTEXT (FAQ relevan):
{context}

FORMAT:
- Jawab singkat dan jelas, maksimal 4 kalimat.
- Di akhir jawaban tuliskan sumbernya, contoh: "Sumber: FAQ #17". Jika informasi tidak ada di FAQ, tulis "Sumber: -".

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

# Lihat contoh prompt
print(build_prompt('apakah bisa bayar pakai cicilan?', semantic_search('apakah bisa bayar pakai cicilan?')))

PERSONA:
Kamu adalah asisten customer service e-commerce yang ramah dan membantu. Gunakan Bahasa Indonesia yang sopan dan mudah dipahami.

TASK:
Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ pada bagian CONTEXT. Jangan menggunakan pengetahuan di luar FAQ dan jangan mengarang informasi.
Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan pengguna untuk menghubungi customer service.

CONTEXT (FAQ relevan):
[FAQ #20]
Pertanyaan: Apakah saya bisa membayar dengan cicilan?
Jawaban: Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

[FAQ #17]
Pertanyaan: Metode pembayaran apa saja yang tersedia?
Jawaban: Kami mendukung transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan.

[FAQ #22]
Pertanyaan: Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
Jawaban: Saat ini sistem belum mendukung pembayaran gabungan (split

## 7. Mengintegrasikan LLM

In [8]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    # Coba ulang beberapa kali jika server Gemini sedang sibuk (error 503 / 429)
    for percobaan in range(5):
        try:
            response = client.models.generate_content(
                model=LLM_MODEL,  # sesuaikan versi model
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.3)
            )
            return response.text, search_results
        except Exception as e:
            if percobaan == 4:
                raise
            print(f'Server sibuk ({e.__class__.__name__}), coba lagi dalam 15 detik...')
            time.sleep(15)

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', f"FAQ #{s['faq_id']}", s['question'], '| distance:', s['distance'])

Jawaban LLM:
Halo! Ya, fitur cicilan tersedia untuk transaksi menggunakan kartu kredit tertentu dengan nominal minimal yang ditentukan oleh pihak bank. 

Sumber: FAQ #20

Sumber FAQ yang digunakan:
- FAQ #20 Apakah saya bisa membayar dengan cicilan? | distance: 0.372
- FAQ #17 Metode pembayaran apa saja yang tersedia? | distance: 0.6044
- FAQ #22 Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? | distance: 0.6168


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [9]:
test_queries = [
    # Query yang jelas ada di FAQ
    'Metode pembayaran apa saja yang tersedia?',                          # harusnya FAQ #17
    'Bagaimana cara membatalkan pesanan?',                                # harusnya FAQ #3
    # Query parafrase / sinonim
    'Barang yang aku terima pecah dan kardusnya penyok, harus gimana?',  # harusnya FAQ #16
    'Kalau beli HP mahal bisa dicicil per bulan nggak?',                  # harusnya FAQ #20
    'Uang saya kapan balik kalau barangnya sudah saya retur?',           # harusnya FAQ #25
    # Query yang tidak ada di FAQ
    'Apakah ada toko offline yang bisa saya kunjungi di Bandung?',
    'Apakah saya bisa membayar menggunakan Bitcoin atau crypto?',
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Hasil retrieval:')
    for s in sources:
        print(f"  {s['rank']}. FAQ #{s['faq_id']} - {s['question']} (distance: {s['distance']})")
    print('---')

Query: Metode pembayaran apa saja yang tersedia?


Jawaban: Halo! Kami mendukung beberapa metode pembayaran, yaitu transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan. 

Sumber: FAQ #17
Hasil retrieval:
  1. FAQ #17 - Metode pembayaran apa saja yang tersedia? (distance: 0.4068)
  2. FAQ #22 - Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? (distance: 0.5411)
  3. FAQ #20 - Apakah saya bisa membayar dengan cicilan? (distance: 0.5736)
---
Query: Bagaimana cara membatalkan pesanan?


Jawaban: Halo! Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status pesanan masih 'Diproses'. Namun, jika status sudah berubah menjadi 'Dikirim', pembatalan tidak dapat dilakukan lagi. 

Sumber: FAQ #3
Hasil retrieval:
  1. FAQ #3 - Bagaimana cara membatalkan pesanan yang sudah dibuat? (distance: 0.4239)
  2. FAQ #24 - Bagaimana prosedur pengembalian barang yang tidak sesuai? (distance: 0.5618)
  3. FAQ #8 - Kenapa pesanan saya otomatis dibatalkan sistem? (distance: 0.5842)
---
Query: Barang yang aku terima pecah dan kardusnya penyok, harus gimana?


Jawaban: Halo Kak! Jika paket diterima dalam kondisi rusak, Kakak harus segera mengambil foto atau video kondisi paket saat diterima. Setelah itu, laporkan melalui menu komplain dalam waktu 2x24 jam sejak barang diterima ya.

Sumber: FAQ #16
Hasil retrieval:
  1. FAQ #16 - Apa yang harus dilakukan jika paket diterima dalam kondisi rusak? (distance: 0.4291)
  2. FAQ #54 - Saya menerima barang yang salah, apa yang harus saya lakukan? (distance: 0.5337)
  3. FAQ #24 - Bagaimana prosedur pengembalian barang yang tidak sesuai? (distance: 0.5784)
---
Query: Kalau beli HP mahal bisa dicicil per bulan nggak?


Jawaban: Halo! Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. 

Sumber: FAQ #20
Hasil retrieval:
  1. FAQ #20 - Apakah saya bisa membayar dengan cicilan? (distance: 0.5024)
  2. FAQ #40 - Bagaimana cara mendapatkan info promo terbaru? (distance: 0.6842)
  3. FAQ #57 - Bagaimana cara membandingkan beberapa produk sekaligus? (distance: 0.6902)
---
Query: Uang saya kapan balik kalau barangnya sudah saya retur?


Jawaban: Halo! Dana refund biasanya akan diproses dalam waktu 3 hingga 7 hari kerja setelah barang yang dikembalikan diterima dan diverifikasi oleh penjual. Dana tersebut akan dikembalikan ke metode pembayaran awal Anda atau ke saldo akun, tergantung pada opsi yang Anda pilih saat pengajuan.

Sumber: FAQ #25 dan #27
Hasil retrieval:
  1. FAQ #25 - Berapa lama proses refund setelah pengembalian disetujui? (distance: 0.4241)
  2. FAQ #27 - Ke mana dana refund akan dikembalikan? (distance: 0.5329)
  3. FAQ #24 - Bagaimana prosedur pengembalian barang yang tidak sesuai? (distance: 0.5436)
---
Query: Apakah ada toko offline yang bisa saya kunjungi di Bandung?


Jawaban: Mohon maaf, saya belum memiliki informasi tersebut. Silakan menghubungi customer service kami untuk mendapatkan bantuan lebih lanjut mengenai hal ini.

Sumber: -
Hasil retrieval:
  1. FAQ #58 - Apakah bisa memesan dalam jumlah besar untuk kebutuhan reseller? (distance: 0.72)
  2. FAQ #46 - Di mana saya bisa melihat ukuran/size chart produk pakaian? (distance: 0.7375)
  3. FAQ #2 - Apakah saya bisa memesan tanpa membuat akun? (distance: 0.7376)
---
Query: Apakah saya bisa membayar menggunakan Bitcoin atau crypto?


Jawaban: Mohon maaf, saat ini kami belum memiliki informasi tersebut di dalam sistem kami. Silakan hubungi customer service kami untuk mendapatkan bantuan lebih lanjut.

Sumber: -
Hasil retrieval:
  1. FAQ #17 - Metode pembayaran apa saja yang tersedia? (distance: 0.6137)
  2. FAQ #18 - Apakah ada biaya tambahan untuk pembayaran menggunakan kartu kredit? (distance: 0.6761)
  3. FAQ #22 - Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? (distance: 0.6872)
---


### Catatan Analisis Relevansi

**Ringkasan:** semua 5 query yang jawabannya ada di FAQ berhasil mendapatkan FAQ yang benar di **peringkat 1 (5/5)**, termasuk 3 query parafrase. Dua query di luar FAQ juga dijawab dengan benar bahwa informasinya belum tersedia (tidak ada halusinasi).

| # | Tipe | Query | FAQ diharapkan | Top-1 (distance) | Hasil |
|---|---|---|---|---|---|
| 1 | Ada di FAQ | Metode pembayaran apa saja yang tersedia? | #17 | #17 (0.41) | ✅ |
| 2 | Ada di FAQ | Bagaimana cara membatalkan pesanan? | #3 | #3 (0.42) | ✅ |
| 3 | Parafrase | Barang ... pecah dan kardusnya penyok, harus gimana? | #16 | #16 (0.43) | ✅ |
| 4 | Parafrase | Kalau beli HP mahal bisa dicicil per bulan nggak? | #20 | #20 (0.50) | ✅ |
| 5 | Parafrase | Uang saya kapan balik kalau barangnya sudah saya retur? | #25 | #25 (0.42) | ✅ |
| 6 | Tidak ada | Toko offline di Bandung? | - | #58 (0.72) | ✅ LLM menolak |
| 7 | Tidak ada | Bayar pakai Bitcoin/crypto? | - | #17 (0.61) | ✅ LLM menolak |

**Query yang retrieval-nya akurat**
- Query 1 dan 2 (wording mirip FAQ) mendapat distance paling kecil (0.41–0.42), sesuai harapan.
- Query parafrase juga berhasil. "pecah dan kardusnya penyok" tidak memakai kata "rusak" sama sekali tetapi tetap mendapat FAQ #16 (paket rusak). "uang kapan balik ... retur" berhasil dicocokkan dengan "refund/pengembalian" (#25). Ini menunjukkan semantic search menangkap **makna**, bukan hanya kata kunci.
- Distance query 4 (0.50) paling besar di antara query valid, karena ada informasi tambahan ("HP mahal", "per bulan") dan bahasa informal ("nggak"). Meski begitu, jaraknya ke peringkat 2 cukup jauh (0.68), jadi hasilnya tetap jelas.

**Hasil yang kurang relevan**
- Peringkat 2–3 sering berisi FAQ yang hanya "bertetangga". Contoh: query 4 mengambil #40 (info promo) dan #57 (bandingkan produk) yang tidak berhubungan. Pada query 2, #24 (pengembalian barang) berada di atas #8 (pesanan dibatalkan otomatis).
- Untuk query di luar FAQ, sistem tetap mengirim 3 FAQ ke LLM karena selalu mengambil top-3. Yang mencegah jawaban ngawur adalah instruksi di prompt, bukan retrieval-nya.
- Terlihat pola distance: query yang ada di FAQ mendapat top-1 **0.40–0.50**, sedangkan query di luar FAQ **0.61–0.72**. Artinya ambang distance sekitar 0.55 bisa dipakai untuk membuang context yang tidak relevan (perlu diuji dengan lebih banyak query).

**Kualitas jawaban LLM**
- Semua jawaban sesuai isi FAQ, singkat, dan mencantumkan sumber (`Sumber: FAQ #...`), sehingga mudah dicek.
- Query 5: LLM menggabungkan FAQ #25 (lama refund) dan #27 (ke mana dana dikembalikan) menjadi satu jawaban yang lebih lengkap. Format sumber sedikit berbeda dari contoh ("FAQ #25 dan #27").
- Query 7 (crypto) adalah kasus paling menarik: FAQ #17 tentang metode pembayaran ikut terambil (distance 0.61), tetapi LLM **tidak mengarang** jawaban "crypto tidak didukung/didukung" dan memilih fallback. Jawaban ini aman, walaupun bisa lebih membantu jika sekaligus menyebutkan metode pembayaran yang tersedia.
- Sapaan belum konsisten ("Halo!" vs "Halo Kak!") karena tidak diatur di prompt.
